In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from torch.utils.data import DataLoader

# Pfadsicherheit
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

# Zentralisierte Imports über deine neuen Schnittstellen
from src.config import Config
from src.data import load_prepared_datasets, process_and_split_data

# 1. Config laden
config = Config()

# 2. Pipeline ausführen (Erzeugt und speichert die Daten)
process_and_split_data(config)

# 3. Datasets von Festplatte laden
train_set, val_set, test_set = load_prepared_datasets(config)

# 4. Dimensionen prüfen
print("\n--- 📊 Verzeichnis- & Split-Validierung ---")
print(f"Train-Sequenzen: {len(train_set)}")
print(f"Val-Sequenzen:   {len(val_set)}")
print(f"Test-Sequenzen:  {len(test_set)}")

# DataLoader-Test
train_loader = DataLoader(train_set, batch_size=config.training["batch_size"], shuffle=False)
batch_x, batch_y = next(iter(train_loader))
print(f"\nBatch X Shape: {batch_x.shape} (Aus Config: Batch={config.training['batch_size']})")

2026-06-14 14:44:27 - INFO - [data_loader.py:38] - Starte Datenaufbereitung und 3-Wege-Split...
2026-06-14 14:44:27 - INFO - [data_loader.py:73] - Split-Verhältnis: Train=2772 | Val=594 | Test=595
2026-06-14 14:44:27 - INFO - [data_loader.py:102] - ✅ Alle Daten erfolgreich in '../data/processed' persistiert.

--- 📊 Verzeichnis- & Split-Validierung ---
Train-Sequenzen: 2760
Val-Sequenzen:   582
Test-Sequenzen:  583

Batch X Shape: torch.Size([64, 12, 7]) (Aus Config: Batch=64)


In [3]:
from src.data import get_data_loaders

# Alle DataLoader über die zentrale Fabrik-Funktion anfordern
train_loader, val_loader, test_loader = get_data_loaders(config)

# Verifikation der Batches
for name, loader in [("Train", train_loader), ("Val", val_loader), ("Test", test_loader)]:
    batch_x, batch_y = next(iter(loader))
    print(f"{name} -> X: {batch_x.shape}, y: {batch_y.shape} | Datentyp: {batch_x.dtype}")

2026-06-14 14:44:27 - INFO - [data_loader.py:132] - Generiere PyTorch DataLoader für Train, Val und Test...
2026-06-14 14:44:27 - INFO - [data_loader.py:145] - ✅ DataLoader erfolgreich erstellt. Batches pro Epoche: Train=43 | Val=10 | Test=10
Train -> X: torch.Size([64, 12, 7]), y: torch.Size([64]) | Datentyp: torch.float32
Val -> X: torch.Size([64, 12, 7]), y: torch.Size([64]) | Datentyp: torch.float32
Test -> X: torch.Size([64, 12, 7]), y: torch.Size([64]) | Datentyp: torch.float32


In [4]:
from src.database.schemas import HubServerSchema, SatServerTelemetrySchema

# 1. Simuliere einen Log-Eintrag
instance = "i-09ab12cd34ef5678a"
cpu_value = 24.5

# 2. Generiere die DV 2.0 Keys
hk = HubServerSchema.generate_hash_key(instance)
diff = SatServerTelemetrySchema.generate_hash_diff(cpu_value, 0.0, 0.0)

print("--- 🏛️ Data Vault 2.0 Token Verification ---")
print(f"Business Key: {instance}")
print(f"➔ Hub Hash Key (SHA-256): {hk}")
print(f"➔ Sat Hash Diff (SHA-256): {diff}")

--- 🏛️ Data Vault 2.0 Token Verification ---
Business Key: i-09ab12cd34ef5678a
➔ Hub Hash Key (SHA-256): 5dd08bb572a40e49c4ac6d39279a3dd7c1bffcb96c83925f166a279fcbfdf108
➔ Sat Hash Diff (SHA-256): a4eaa2b76e1810d9c6632c5952c9f126ebab89cb6796c1752cdd6780f5c6d877


In [5]:
from src.config import Config
from src.data.spark_pipeline import run_spark_ingestion
from pyspark.sql import SparkSession
from pathlib import Path

# 1. Config laden und Spark Ingestion ausführen
config = Config()
run_spark_ingestion(config)

# 2. Lokale Spark Session im Notebook öffnen, um die geschriebenen Parquet-Dateien zu prüfen
spark = SparkSession.builder.appName("NotebookValidation").getOrCreate()

project_root = Path.cwd().parent
hub_path = project_root / "data" / "data_vault" / "hub_server"
sat_path = project_root / "data" / "data_vault" / "sat_server_telemetry"

print("\n" + "="*50)
print("🔎 INTERAKTIVE DATA VAULT 2.0 VALIDIERUNG")
print("="*50)

# 3. HUB_SERVER prüfen
print("\n📊 HUB_SERVER SCHEMA & BEISPIEL-DATEN:")
df_hub = spark.read.parquet(str(hub_path))
df_hub.printSchema()
df_hub.show(truncate=False)

# 4. SAT_SERVER_TELEMETRY prüfen
print("\n📊 SAT_SERVER_TELEMETRY SCHEMA & INFERENZ-PREVIEW:")
df_sat = spark.read.parquet(str(sat_path))
df_sat.printSchema()
# Zeige die ersten 5 Zeilen sortiert nach Timestamp
df_sat.orderBy("timestamp").show(5, truncate=False)

# Spark Session für das Notebook sauber schließen
spark.stop()

2026-06-14 14:50:39 - INFO - [spark_pipeline.py:14] - Initialisiere Apache Spark Session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 14:50:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-06-14 14:50:59 - INFO - [spark_pipeline.py:36] - Lade Rohdaten via Spark von: /Users/rodrigue.lawson/VSCode Projects/infra-insight/data/external/aws_cpu_utilization.csv
2026-06-14 14:51:00 - INFO - [spark_pipeline.py:52] - Transformiere Rohdaten in Data Vault 2.0 Strukturen...
2026-06-14 14:51:02 - INFO - [spark_pipeline.py:78] - 💾 Hub erfolgreich persistiert unter: /Users/rodrigue.lawson/VSCode Projects/infra-insight/data/data_vault/hub_server
2026-06-14 14:51:02 - INFO - [spark_pipeline.py:79] - 💾 Satellit erfolgreich persistiert unter: /Users/rodrigue.lawson/VSCode Projects/infra-insight/data/data_vault/sat_server_telemetry
2026-06-14 14:51:03 - INFO - [spark_pipeline.py:82] - Spark Session erfolgreich geschlossen.

🔎 INTERAKTIVE DATA VAULT 2.0 VALIDIERUNG

📊 HUB_SERVER SCHEMA & BEISPIEL-DATEN:
root
 |-- hk_server: string (nullable = true)
 |-- instance_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- record_source: string (nullable = true)

+-

In [7]:
from src.data.spark_pipeline import run_spark_feature_engineering
from pyspark.sql import SparkSession
from pathlib import Path

# 1. Zünde die neue Feature-Pipeline
run_spark_feature_engineering()

# 2. Prüfe das mathematische Ergebnis
spark = SparkSession.builder.appName("FeatureValidation").getOrCreate()
project_root = Path.cwd().parent
feature_path = project_root / "data" / "processed_spark" / "features.parquet"

df_feat = spark.read.parquet(str(feature_path))

print("\n" + "="*50)
print("🔎 MULTIVARIATE FEATURE ENGINNERING VALIDATION")
print("="*50)
df_feat.printSchema()

# Zeige die berechneten Features für die ersten Zeilen
df_feat.select(
    "timestamp", "cpu_utilization", "hour_sin", "rolling_mean_1h", "rolling_std_1h", "rolling_mean_6h"
).orderBy("timestamp").show(5, truncate=False)

spark.stop()

2026-06-14 18:15:22 - INFO - [spark_pipeline.py:15] - Initialisiere Apache Spark Session...
2026-06-14 18:15:22 - INFO - [spark_pipeline.py:73] - Lade Satellitendaten für Feature Engineering aus: /Users/rodrigue.lawson/VSCode Projects/infra-insight/data/data_vault/sat_server_telemetry
2026-06-14 18:15:22 - INFO - [spark_pipeline.py:80] - Berechne zyklische Zeit-Features (Stunde & Wochentag) und rollierende Statistiken...
2026-06-14 18:15:22 - INFO - [spark_pipeline.py:109] - 📊 Generiere Spark Execution Plan (Physischer Ausführungsplan):
== Physical Plan ==
AdaptiveSparkPlan (12)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- AdaptiveSparkPlan (11)
               +- Filter (10)
                  +- Window (9)
                     +- Sort (8)
                        +- Exchange (7)
                           +- Project (6)
                              +- Project (5)
                                 +- Project (4)
                                    +- Scan parquet